# Homework: Computational Complexity

## Problem 1: Complexity Analysis

For each of the following code snippets, determine the time complexity and space complexity in Big-O notation. Provide a brief justification for each.

**(a)**

In [ ]:
def func_a(data):
    n = len(data)
    total = 0
    for i in range(0, n, 2):
        for j in range(5):
            total += data[i]
    return total

O(n), as the second loop is constant 5.

**(b)**

In [ ]:
def func_b(matrix):
    n = len(matrix)
    result = []
    for i in range(n):
        for j in range(i):
            result.append(matrix[i][j])
    return result

o(n^2), this is the lower triangle part of the matrix.

**(c)**

In [ ]:
def func_c(n):
    if n <= 1:
        return 1
    return func_c(n // 2) + func_c(n // 2)

O(n), as each call creates two recursive calls of size n/2.
**(d)**

In [ ]:
def func_d(items):
    seen = set()
    duplicates = []
    for item in items:
        if item in seen:
            duplicates.append(item)
        seen.add(item)
    return duplicates

O(n), as the loop will run the length of the items times.

## Problem 2: Choosing the Right Data Structure

Finding elements common to two collections is a frequent operation in data processing (for example, finding shared gene IDs across two experiments).

**(a)** The following function finds common elements using a naive approach. What is its time complexity if `list1` has n elements and `list2` has m elements? Explain why.

In [2]:
def find_common_naive(list1, list2):
    common = []
    for x in list1:
        for y in list2:
            if x == y and x not in common:
                common.append(x)
    return common

o(mn), this code will loop all the m by n elements in x and y list.

**(b)** Write an efficient version called `find_common_fast` that uses a set to achieve better time complexity. What is the time complexity of your version? For example, `find_common_fast([1, 2, 3, 4], [3, 4, 5, 6])` should return `[3, 4]` (in any order).

In [1]:
def find_common_fast(list1, list2):
    set2 = set(list2)
    common = []

    for x in list1:
        if x in set2 and x not in common:
            common.append(x)

    return common

**(c)** Verify the speedup empirically. Write a script that times both functions on lists of size n = 1000, 5000, 10000, and 20000 (where elements are random integers from 0 to n). Print the time for each function at each size.


In [3]:
import random
import time


def find_common_naive(list1, list2):
    common = []
    for x in list1:
        for y in list2:
            if x == y and x not in common:
                common.append(x)
    return common


def find_common_fast(list1, list2):
    set2 = set(list2)
    common = []
    for x in list1:
        if x in set2 and x not in common:
            common.append(x)
    return common


sizes = [1000, 5000, 10000, 20000]

for n in sizes:
    list1 = [random.randint(0, n) for _ in range(n)]
    list2 = [random.randint(0, n) for _ in range(n)]

    start = time.time()
    find_common_naive(list1, list2)
    naive_time = time.time() - start

    start = time.time()
    find_common_fast(list1, list2)
    fast_time = time.time() - start

    print(f"n={n}")
    print(f"naive: {naive_time:.4f} seconds")
    print(f"fast:  {fast_time:.4f} seconds")
    print()

n=1000
naive: 0.0267 seconds
fast:  0.0015 seconds

n=5000
naive: 0.5322 seconds
fast:  0.0395 seconds

n=10000
naive: 3.5235 seconds
fast:  0.1333 seconds

n=20000
naive: 8.3848 seconds
fast:  0.5257 seconds




## Problem 3: Empirical Complexity Measurement

The following function performs a computation on a list:


In [4]:
def mystery_function(data):
    n = len(data)
    data = sorted(data)
    total = 0
    for i in range(n):
        left, right = 0, n - 1
        while left < right:
            if data[left] + data[right] == data[i]:
                total += 1
            if data[left] + data[right] < data[i]:
                left += 1
            else:
                right -= 1
    return total

Your task is to empirically determine the time complexity of `mystery_function` using the log-log method from the lecture.

**(a)** Write a timing script that measures the runtime of `mystery_function` for input sizes n = 500, 1000, 2000, 4000, and 8000. Use random integer data for each size. Run each size at least 3 times and take the average.

In [5]:
import numpy as np
from scipy.stats import linregress

def time_one_n(n, reps=3, seed=0):
    rng = random.Random(seed + n)
    times = []
    for r in range(reps):
        data = [rng.randint(0, n) for _ in range(n)]
        t0 = time.perf_counter()
        mystery_function(data)
        t1 = time.perf_counter()
        times.append(t1 - t0)
    return float(np.mean(times))

sizes = [500, 1000, 2000, 4000, 8000]
avg_times = [time_one_n(n, reps=3) for n in sizes]

for n, t in zip(sizes, avg_times):
    print(f"n={n:5d}  avg_time={t:.6f} sec")

n=  500  avg_time=0.072779 sec
n= 1000  avg_time=0.301026 sec
n= 2000  avg_time=0.706684 sec
n= 4000  avg_time=2.714515 sec
n= 8000  avg_time=15.468219 sec


(b) Compute the log-log slope using scipy.stats.linregress on the log of the sizes and the log of the times. Report the slope value.

In [6]:
log_n = np.log(sizes)
log_t = np.log(avg_times)

slope, intercept, r, pvalue, stderr = linregress(log_n, log_t)

print(f"log-log slope: {slope:.3f}")
print(f"R^2: {r*2:.4f}")

log-log slope: 1.864
R^2: 1.9879


(c) Based on the slope, what is the time complexity of mystery_function? Explain why this matches (or doesn't match) what you would expect from reading the code.

The slope is close to 2, implying this is O(n^2).

Problem 4: Improving a Statistical Computation
In Bayesian statistics and spatial statistics, you often need to solve many linear systems with the same coefficient matrix but different right-hand side vectors. For example, drawing samples from a multivariate normal or computing conditional distributions.

The following function solves m linear systems using a loop:



In [7]:
import numpy as np


def solve_systems_naive(A, B):
    """Solve A @ X = B for X, where B has m columns.

    Parameters
    ----------
    A : np.ndarray
        Symmetric positive definite matrix of shape (n, n).
    B : np.ndarray
        Matrix of shape (n, m), each column is a right-hand side vector.

    Returns
    -------
    np.ndarray
        Solution matrix X of shape (n, m).
    """
    n, m = B.shape
    X = np.zeros_like(B)
    for i in range(m):
        X[:, i] = np.linalg.solve(A, B[:, i])
    return X

**(a)** What is the time complexity of `solve_systems_naive` in terms of n and m? Explain what happens inside `np.linalg.solve` on each iteration.

O(mn^3). The LU decomp is O(n^3), and tehre are m loops, so it is mn^3

**(b)** Rewrite the function as `solve_systems_cholesky` using `scipy.linalg.cholesky` and `scipy.linalg.cho_solve` to factor A once and then solve each system cheaply. What is the new time complexity? For example, `solve_systems_cholesky(np.array([[2, 1], [1, 2]]), np.array([[1, 0], [0, 1]]))` should return `np.array([[2/3, -1/3], [-1/3, 2/3]])` (the inverse of A, since B is the identity).

In [8]:
import scipy.linalg


def solve_systems_cholesky(A, B):
    # Cholesky factorization
    L = scipy.linalg.cholesky(A, lower=True)

    # Solve A X = B using the factorization
    X = scipy.linalg.cho_solve((L, True), B)

    return X

The new complexity is O(n^3 + mn^2)

**(c)** Time both approaches with n = 300 and m = 100, and report the speedup. Use a random symmetric positive definite matrix (for example, `A = Z @ Z.T + n * np.eye(n)` where `Z` is random).


In [9]:

# problem sizes
n = 300
m = 100

Z = np.random.randn(n, n)
A = Z @ Z.T + n * np.eye(n)

B = np.random.randn(n, m)

# time naive
start = time.time()
solve_systems_naive(A, B)
naive_time = time.time() - start

# time cholesky
start = time.time()
solve_systems_cholesky(A, B)
chol_time = time.time() - start

speedup = naive_time / chol_time

print(f"Naive time: {naive_time:.4f} sec")
print(f"Cholesky time: {chol_time:.4f} sec")
print(f"Speedup: {speedup:.2f}x")

Naive time: 0.2994 sec
Cholesky time: 0.0279 sec
Speedup: 10.73x



## Problem 5: Optimizing Pairwise Computation

The following function computes the sum of all pairwise absolute differences in an array:

$$S = \sum_{i=0}^{n-1} \sum_{j=i+1}^{n-1} |a_i - a_j|$$

In [ ]:
def pairwise_abs_diff_slow(arr):
    """Compute sum of all pairwise absolute differences.

    Parameters
    ----------
    arr : list
        A list of numbers.

    Returns
    -------
    float
        Sum of |a_i - a_j| for all pairs i < j.

    Examples
    --------
    >>> pairwise_abs_diff_slow([1, 2, 4])
    6
    >>> pairwise_abs_diff_slow([2, 8, 4, 6])
    20
    """
    n = len(arr)
    total = 0
    for i in range(n):
        for j in range(i + 1, n):
            total += abs(arr[i] - arr[j])
    return total

This runs in O(n^2) time. Your task is to write a function `pairwise_abs_diff_fast` that computes the same result in O(n log n) time.

In [10]:
def pairwise_abs_diff_fast(arr):
    arr = sorted(arr)
    prefix_sum = 0
    total = 0

    for i, x in enumerate(arr):
        total += i * x - prefix_sum
        prefix_sum += x

    return total

Hint: consider what happens when you sort the array first. After sorting, every element `a[k]` is greater than or equal to all elements before it. Think about how many times `a[k]` is added versus subtracted across all pairs that include index k.